# Regression: your DataFrame, a reusable predictor

SDK 0.7. Download this notebook and its matching support.py into the same directory. Restart/run all. Default mode is controlled synthetic HTTP transport, not a foundation model or scientific benchmark. Explicit live mode makes a hosted fit/predict and retains its synthetic state. No fallback, deletion or saved prediction output. The frozen recipe is seed42, 128 training rows, four features and 32 query rows. Read README.md before live use.


## Build separate training and query tables

The training table includes the explicit target. Query rows include features only. The helper provides the documented frozen recipe and keeps fixture plumbing out of the public estimator API.


In [ ]:
from support import estimator_transport, regression_table
import pandas as pd
import numpy as np
from welt import Regressor

columns, values, target, query_values = regression_table()
train = pd.DataFrame(values, columns=columns)
train["target"] = target
query = pd.DataFrame(query_values, columns=columns)


## Fit, predict and reopen

Choose `Regressor` explicitly. `fit(train, target="target")` excludes the named target from features. Fit uploads and waits for remote preparation; prediction executes remotely. In default mode the context below installs only the controlled transport. In live mode the same public SDK calls run against the configured service.


In [ ]:
with estimator_transport():
    model = Regressor(model="tabicl-v2", random_state=42)
    model.fit(train, target="target")
    predictions = model.predict(query)
    assert predictions.shape == (32,)
    pinned_version = model.model_version_
    reopened = Regressor.from_predictor(model.predictor_id_)
    repeat = reopened.predict(query[columns[::-1]])
    assert reopened.model_version_ == pinned_version
    assert np.isfinite(predictions).all()
    np.testing.assert_allclose(predictions, repeat, atol=1e-5, rtol=1e-5)
    details = model.predict_details(query)
    assert details["classes"] is None and details["probabilities"] is None
print(type(predictions).__name__, predictions.shape)


## Read the result

Expected output contract: `ndarray (32,)`, one result per query row in order. Classification probabilities use `classes_` order and are not calibrated confidence. Regression returns finite mean points in target units, without class probabilities or calibrated intervals. Reopening preserves task/model/configuration/version/seed and makes no new fit or training upload; this does not force a worker restart. Keep `predictor_id_` as experiment metadata. Local timeout preserves durable work: reconnect by job ID instead of refitting. Sources contain no execution results.
